# AstroCLIP Teaching Lab

This notebook walks through a minimal, end-to-end multimodal pipeline:

1. Prepare a manageable subset of the hosted `EiffL/AstroCLIP` dataset.
2. Train lightweight image and spectrum encoders from scratch.
3. Align the two modalities with a CLIP-style objective.
4. Explore the shared embedding space and perform a simple downstream regression task.

The notebook mirrors the Python scripts in `teaching_scripts/`, so you can either run the cells here or launch the CLI equivalents on the HPC cluster.

## 0. Environment and Data

Make sure you have activated the `astroclip-teaching` conda environment, exported `ASTROCLIP_ROOT`, and pointed the Hugging Face cache to the location where you staged the dataset (for example `/n03data/huertas/hf_cache`).

In [ ]:
import os
from pathlib import Path

import datasets
from datasets import load_dataset, load_from_disk
import torch

ASTROCLIP_ROOT = Path(os.environ.get('ASTROCLIP_ROOT', '.')).resolve()
DATASET_DIR = ASTROCLIP_ROOT / 'teaching_demo' / 'astroclip_subset'
print(f'ASTROCLIP_ROOT = {ASTROCLIP_ROOT}')
print(f'DATASET_DIR   = {DATASET_DIR}')

### Download / Stage a Subset

Run the subset script once (CLI preferred on the server). Adjust the sample sizes to suit your GPU budget:

```bash
python teaching_scripts/prepare_dataset.py     --dataset EiffL/AstroCLIP     --train-size 5000     --test-size 1000     --output-dir $ASTROCLIP_ROOT/teaching_demo/astroclip_subset     --overwrite
```

If you already ran the command, just load the dataset below.

In [ ]:
if DATASET_DIR.exists():
    ds = load_from_disk(str(DATASET_DIR))
else:
    ds = load_dataset('EiffL/AstroCLIP')

print(ds)
if isinstance(ds, datasets.DatasetDict):
    print({split: len(ds[split]) for split in ds})
elif isinstance(ds, datasets.Dataset):
    print(f'Total samples: {len(ds)}')

## 1. Inspect a Mini-batch

Take a quick look at the image/spectrum pairs to verify the data flow.

In [ ]:
from astroclip.data.datamodule import AstroClipCollator
import matplotlib.pyplot as plt

collator = AstroClipCollator(center_crop=144)

if isinstance(ds, datasets.DatasetDict):
    samples = [ds['train'][i] for i in range(4)]
else:
    samples = [ds[i] for i in range(4)]

sample_batch = collator(samples)
images = sample_batch['image']
spectra = sample_batch['spectrum']

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for i, ax in enumerate(axes):
    ax.imshow(images[i].permute(1, 2, 0).numpy())
    ax.axis('off')
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(spectra[0].view(-1).numpy())
plt.xlabel('Pixel')
plt.ylabel('Flux')
plt.title('Example spectrum')
plt.show()

## 2. Train the Image Encoder

We use a lightweight convolutional autoencoder (`ImageAutoencoder` in `teaching_scripts/models.py`).
Fill in the TODOs to understand the training loop, or run the CLI script:

```bash
python teaching_scripts/train_image_encoder.py     --dataset $ASTROCLIP_ROOT/teaching_demo/astroclip_subset     --output $ASTROCLIP_ROOT/teaching_demo/image_autoencoder.ckpt     --max-epochs 10
```

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint

from teaching_scripts.models import ImageAutoencoder
from teaching_scripts.data_utils import build_image_dataloader

if isinstance(ds, datasets.DatasetDict):
    image_train_loader = build_image_dataloader(ds['train'], batch_size=128, shuffle=True, num_workers=4)
    image_val_loader = build_image_dataloader(ds['test'], batch_size=128, shuffle=False, num_workers=4) if 'test' in ds else None
else:
    image_train_loader = build_image_dataloader(ds, batch_size=128, shuffle=True, num_workers=4)
    image_val_loader = None

image_model = ImageAutoencoder(embed_dim=256)
checkpoint = ModelCheckpoint(dirpath=ASTROCLIP_ROOT / 'teaching_demo', filename='image_autoencoder', save_last=True, save_top_k=1)
trainer = Trainer(accelerator='auto', devices='auto', max_epochs=10, callbacks=[checkpoint])
# TODO: uncomment to train inside the notebook (optional)
# trainer.fit(image_model, image_train_loader, image_val_loader)


## 3. Train the Spectrum Encoder

Analogous to the image stage, but using a fully-connected autoencoder. CLI equivalent:

```bash
python teaching_scripts/train_spectrum_encoder.py     --dataset $ASTROCLIP_ROOT/teaching_demo/astroclip_subset     --output $ASTROCLIP_ROOT/teaching_demo/spectrum_autoencoder.ckpt     --max-epochs 15
```

In [ ]:
from teaching_scripts.models import SpectrumAutoencoder
from teaching_scripts.data_utils import build_spectrum_dataloader

if isinstance(ds, datasets.DatasetDict):
    spectrum_train_loader = build_spectrum_dataloader(ds['train'], batch_size=256, shuffle=True, num_workers=4)
    spectrum_val_loader = build_spectrum_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=4) if 'test' in ds else None
else:
    spectrum_train_loader = build_spectrum_dataloader(ds, batch_size=256, shuffle=True, num_workers=4)
    spectrum_val_loader = None

sample_flat_dim = spectrum_train_loader.dataset[0].numel() if hasattr(spectrum_train_loader.dataset, '__getitem__') else 7781
spectrum_model = SpectrumAutoencoder(input_dim=sample_flat_dim, embed_dim=256)
# TODO: run Trainer here if you prefer not to use the CLI script.


## 4. Align the Modalities (CLIP Objective)

Load the trained autoencoders and optimise the CLIP loss. CLI version:

```bash
python teaching_scripts/train_clip_alignment.py     --dataset $ASTROCLIP_ROOT/teaching_demo/astroclip_subset     --image-ckpt $ASTROCLIP_ROOT/teaching_demo/image_autoencoder.ckpt     --spectrum-ckpt $ASTROCLIP_ROOT/teaching_demo/spectrum_autoencoder.ckpt     --output $ASTROCLIP_ROOT/teaching_demo/clip_alignment.ckpt     --max-epochs 10
```

In [ ]:
from teaching_scripts.train_clip_alignment import main as clip_cli_main

image_ckpt = ASTROCLIP_ROOT / 'teaching_demo' / 'image_autoencoder.ckpt'
spectrum_ckpt = ASTROCLIP_ROOT / 'teaching_demo' / 'spectrum_autoencoder.ckpt'
clip_ckpt = ASTROCLIP_ROOT / 'teaching_demo' / 'clip_alignment.ckpt'

print(f'Image encoder ckpt:    {image_ckpt}')
print(f'Spectrum encoder ckpt: {spectrum_ckpt}')
print(f'CLIP alignment ckpt:   {clip_ckpt}')
# TODO: run the CLI command above or integrate the Trainer here.


## 5. Embed and Explore the Joint Space

Once alignment training finishes, embed a validation split and visualise the embeddings. CLI helper:

```bash
python teaching_scripts/embed_dataset.py     --dataset $ASTROCLIP_ROOT/teaching_demo/astroclip_subset     --clip-ckpt $ASTROCLIP_ROOT/teaching_demo/clip_alignment.ckpt     --image-ckpt $ASTROCLIP_ROOT/teaching_demo/image_autoencoder.ckpt     --spectrum-ckpt $ASTROCLIP_ROOT/teaching_demo/spectrum_autoencoder.ckpt     --split test     --output $ASTROCLIP_ROOT/teaching_demo/embeddings.h5
```

In [ ]:
import numpy as np
import h5py
import umap
import matplotlib.pyplot as plt

emb_path = ASTROCLIP_ROOT / 'teaching_demo' / 'embeddings.h5'
if emb_path.exists():
    with h5py.File(emb_path, 'r') as f:
        image_emb = f['image_embeddings'][:]
        spectrum_emb = f['spectrum_embeddings'][:]
        redshift = f['redshift'][:]

    reducer = umap.UMAP(random_state=42)
    joint = reducer.fit_transform(np.concatenate([image_emb, spectrum_emb], axis=0))
    labels = np.concatenate([np.zeros(len(image_emb)), np.ones(len(spectrum_emb))])

    plt.figure(figsize=(6, 5))
    plt.scatter(joint[labels == 0, 0], joint[labels == 0, 1], s=5, alpha=0.5, label='Images')
    plt.scatter(joint[labels == 1, 0], joint[labels == 1, 1], s=5, alpha=0.5, label='Spectra')
    plt.legend()
    plt.title('Joint UMAP embedding')
    plt.show()
else:
    print(f"Embedding file not found at {emb_path}. Run embed_dataset.py first.")


## 6. Downstream Redshift Regression

Train a tiny MLP on the embeddings. CLI:

```bash
python teaching_scripts/downstream_redshift.py     --embeddings $ASTROCLIP_ROOT/teaching_demo/embeddings.h5     --model joint     --epochs 50     --output $ASTROCLIP_ROOT/teaching_demo/redshift_regressor.pt
```

## 7. Next Steps

- Swap in larger subsets or pretrained encoders to see how performance scales.
- Follow `downstream_tasks/morphology_classification/` to add Galaxy Zoo labels once a position bridge becomes available.
- Compare against the scripts in `teaching_scripts/` to understand how the HPC workflow maps onto these notebook steps.